# ERDDAP catalog explorer

`earthlens.erddap` reaches many public ERDDAP servers from one backend. A curated catalog pins each dataset to a concrete `(server_url, dataset_id, protocol)`; the `protocol` decides the output shape — `griddap` -> raster NetCDF, `tabledap` -> tabular `DataFrame`. This notebook is offline: it only reads the bundled catalog.

## Open the catalog

In [1]:
from earthlens.erddap import Catalog

cat = Catalog()
len(cat)

4

## Split by protocol

The protocol is what fixes a dataset's `OUTPUT_KIND` (and therefore whether `aggregate=` is accepted).

In [2]:
by_protocol = {}
for key, ds in cat.datasets.items():
    by_protocol.setdefault(ds.protocol, []).append(key)

for protocol, keys in sorted(by_protocol.items()):
    print(f'{protocol:9s} ({len(keys)}): {keys}')

griddap   (3): ['erdMH1chla8day', 'nceiPH53sstd1day', 'NOAA_DHW']
tabledap  (1): ['cwwcNDBCMet']


## Inspect a row

Each row is self-describing — server, id, protocol, default variables, and (for griddap) the grid dimension order.

In [3]:
for key in ['NOAA_DHW', 'cwwcNDBCMet']:
    ds = cat.get(key)
    print(key)
    print('  server  :', ds.server_url)
    print('  protocol:', ds.protocol)
    print('  vars    :', ds.variables)
    if ds.protocol == 'griddap':
        print('  dims    :', ds.dim_names)
    print('  title   :', ds.title)

NOAA_DHW
  server  : https://coastwatch.pfeg.noaa.gov/erddap
  protocol: griddap
  vars    : ['CRW_SSTANOMALY', 'CRW_DHW']
  dims    : ['time', 'latitude', 'longitude']
  title   : NOAA Coral Reef Watch Daily Global 5km SST anomaly and Degree Heating Weeks
cwwcNDBCMet
  server  : https://coastwatch.pfeg.noaa.gov/erddap
  protocol: tabledap
  vars    : ['station', 'time', 'wtmp']
  title   : NDBC Standard Meteorological Buoy Data


## Reach it through the facade

Users do not import the subpackage — the unified `EarthLens` facade resolves the `erddap` key (alias `ioos`).

In [4]:
from earthlens.earthlens import EarthLens

print('erddap' in EarthLens.DataSources)
print('ioos' in EarthLens.DataSources)

True
True


## Unknown ids raise with a did-you-mean hint

In [5]:
try:
    cat.get('NOAA_DH')
except ValueError as exc:
    print(exc)

'NOAA_DH' is not in the ERDDAP catalog. Known datasets: ['NOAA_DHW', 'cwwcNDBCMet', 'erdMH1chla8day', 'nceiPH53sstd1day']. Did you mean 'NOAA_DHW'?
